# DDL, keys, and constraints

A relational database is only as trustworthy as the rules it enforces. **DDL** (data definition
language) is the part of SQL that creates and changes those rules: tables, primary keys, foreign
keys, uniqueness, checks, and indexes. This notebook builds and reshapes a small schema, then
inspects what the database actually stored.

## Learning objectives

By the end of this notebook you will be able to:

- create tables with `CREATE TABLE` and choose appropriate column types;
- apply `PRIMARY KEY`, `FOREIGN KEY`, `UNIQUE`, `NOT NULL`, `DEFAULT`, and `CHECK` constraints;
- alter a table with `ALTER TABLE ... ADD COLUMN` and remove objects with `DROP`;
- explain why SQLite has no `TRUNCATE` and what to use instead;
- inspect indexes and the schema through `PRAGMA` and `sqlite_master`.

## Concept

A **schema** is the shape of the data plus the rules about it. DDL is how we declare that shape:
`CREATE TABLE` defines columns and their types; constraints say what values are allowed. A
**primary key** uniquely identifies a row. A **foreign key** says a value must exist in another
table. `UNIQUE` forbids duplicates, `NOT NULL` requires a value, `DEFAULT` supplies one, and
`CHECK` bounds it. Together these rules push correctness into the database instead of trusting
every application that writes to it.

Types matter less in SQLite than in PostgreSQL — it is dynamically typed and stores what you
give it — but declaring `INTEGER`, `TEXT`, and `REAL` documents intent and enables `CHECK`s.
`ALTER TABLE` can add a column or rename things, but dropping a constraint is not supported, so
schema changes are often done by creating a new table and copying rows. SQLite also has no
`TRUNCATE`; the equivalent is `DELETE FROM table` (optionally inside a transaction).

**Indexes** are separate lookup structures that make `WHERE` and `JOIN` fast at the cost of
storage and slower writes. A primary key is indexed automatically; other columns need an explicit
`CREATE INDEX`.

This module ships a normalised schema in `schema.sql` covering countries, cities, airports,
airlines, and routes. We use it as the reference model throughout.

## Worked example

### Set up a scratch database and apply the real schema

We open an in-memory database so nothing is written to disk, then run the module's `schema.sql`.
`conn.executescript` runs several statements at once.

In [ ]:
import sqlite3
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from load import SCHEMA_PATH

conn = sqlite3.connect(":memory:")
conn.executescript(SCHEMA_PATH.read_text(encoding="utf-8"))
print("applied:", SCHEMA_PATH.name)

### What did the database store?

`schema.sql` creates five tables and one view. `sqlite_master` is the catalogue that lists every
object, and `PRAGMA foreign_key_list` reports the references a table declares.

In [ ]:
objects = conn.execute(
    "SELECT type, name FROM sqlite_master WHERE name NOT LIKE 'sqlite_%' ORDER BY type, name"
).fetchall()
for kind, name in objects:
    print(f"{kind:<7} {name}")

In [ ]:
print("foreign keys on routes:")
for row in conn.execute("PRAGMA foreign_key_list(routes)"):
    print("  ", row)
print("\nindexes on airports:")
for row in conn.execute("PRAGMA index_list(airports)"):
    print("  ", row)

### Constraints in action

The primary key rejects duplicate ids, `CHECK` rejects impossible latitudes, and a foreign key
rejects a route that points at an airport that does not exist. Foreign keys must be switched on
per connection with `PRAGMA foreign_keys = ON`.

In [ ]:
conn.execute("PRAGMA foreign_keys = ON")

def attempt(sql, params=()):
    try:
        conn.execute(sql, params)
        conn.commit()
        print("accepted:", sql.split("(")[0].strip())
    except sqlite3.IntegrityError as exc:
        print("rejected:", exc)

attempt("INSERT INTO countries (country_id, name) VALUES (1, 'Iceland')")
attempt("INSERT INTO countries (country_id, name) VALUES (2, 'Iceland')")   # duplicate id
attempt("INSERT INTO countries (country_id, name) VALUES (3, 'Nowhere')")   # duplicate name
attempt("INSERT INTO airports (airport_id, name, latitude, longitude) "
        "VALUES (1, 'Bad Latitude', 120, 0)")                                # CHECK fails
attempt("INSERT INTO routes (source_airport_id, destination_airport_id) "
        "VALUES (999, 998)")                                                 # missing FK

### Widening a table

`ALTER TABLE ... ADD COLUMN` adds a column to every existing row with its default. Here we record
the elevation band a country's average airport sits at. Adding a column is cheap; changing a
constraint is not, which is why we plan the schema before loading data.

In [ ]:
conn.execute("ALTER TABLE countries ADD COLUMN hemisphere TEXT DEFAULT 'unknown'")
print([row[1] for row in conn.execute("PRAGMA table_info(countries)")])

conn.execute("UPDATE countries SET hemisphere = 'north'")
print(conn.execute("SELECT * FROM countries").fetchall())

### Removing objects

`DROP TABLE` and `DROP VIEW` remove objects permanently. There is no `TRUNCATE`; the closest is
`DELETE FROM`, which keeps the table but removes its rows.

In [ ]:
rows_before = conn.execute("SELECT COUNT(*) FROM countries").fetchone()[0]
conn.execute("DELETE FROM countries")
conn.commit()
print(f"DELETE removed {rows_before} rows; table still exists:",
      conn.execute("SELECT COUNT(*) FROM countries").fetchone()[0] == 0)

conn.executescript("DROP VIEW IF EXISTS v_route_details; DROP TABLE IF EXISTS routes;")
print("remaining objects:",
      [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")])

conn.close()

## Exercises

1. **Add an index.** Create a scratch database, apply `schema.sql`, then add an index on
   `airports (name)`. Confirm it exists with `PRAGMA index_list('airports')`.
2. **Enforce a range.** Write a new `CREATE TABLE measurements (id INTEGER PRIMARY KEY,
   flipper_mm INTEGER NOT NULL CHECK (flipper_mm BETWEEN 100 AND 400))`, insert three rows (one
   out of range), and show which insert the database rejects.
3. **Drop vs delete.** Explain in two sentences the difference between `DROP TABLE` and
   `DELETE FROM`, then demonstrate both on a small table.

## Limitations

SQLite is deliberately lightweight: it has little type enforcement, no `TRUNCATE`, no dropping
of individual constraints, and no `RIGHT JOIN` before recent versions. Production PostgreSQL or
MySQL enforce types far more strictly and let you alter constraints in place. Foreign keys are
off by default in SQLite, so a connection that forgets `PRAGMA foreign_keys = ON` will happily
store dangling references. Finally, indexes speed reads but slow every insert, so the right
number of indexes is a trade-off rather than "as many as possible".